# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and performing basic analysis on the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. This dataset contains protein abundance and metadata for human extracellular vesicles, described by a Croissant schema.

### Dataset Source
The dataset Croissant schema is available at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and access records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")

## 2. Data Overview
List available record sets and their fields using their unique `@id`. This helps in referencing them for data extraction and deeper analysis.

In [ ]:
# Show available record sets and their fields (@ids)
print("Record sets found in the Croissant metadata:")
record_sets = []
if hasattr(metadata, 'recordSet'):
    for rs in metadata.recordSet:
        print(f"- Record set @id: {rs['@id']}")
        record_sets.append(rs['@id'])
        if 'field' in rs:
            print("    Fields / columns with @id:")
            for field in rs['field']:
                # If the field is a dict with @id
                field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
                print(f"        - {field_id}")
else:
    print("(No direct recordSet list found; checking distribution for tabular data sources)")
    # Explore likely tabular sources from distribution > contentUrl
    if hasattr(metadata, 'distribution'):
        for d in metadata.distribution:
            print(f" Distribution @id: {d['@id']} contentUrl: {d.get('contentUrl', '')}")

## 3. Data Extraction
Load data from a record set into a pandas DataFrame. 

Below, we select a record set by its `@id` and extract records. To find proper `@id`, consult the output above (most datasets have one central record set with fields like 'Accession', 'Description', 'MW', etc.).

In [ ]:
# If manual inspection in the previous cell didn't show recordSet, try this default (well-known for this dataset as of 2024-06):
# For FAIR2, the main record set usually starts with 'cr:RecordSet' and has fields describing protein entries.
if not record_sets:
    # Fallback: try to infer or hardcode if necessary
    # For Croissant generated by SenScience, typically recordSet['@id'] looks like 'cr:protein_set'
    record_sets = ['cr:protein_set']

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Fields/columns in {record_set_id}:")
    print(df.columns.tolist())
    print(df.head())

# For this dataset, use the main record set for analysis
main_record_set = record_sets[0]
main_df = dataframes[main_record_set]

## 4. Exploratory Data Analysis (EDA)
Let's analyze protein mass (`MW`), which is a numeric field in most proteomics datasets. We'll filter proteins with MW > 10000 Da, normalize, and group by another field (e.g., 'Description'). All fields referenced by their exact `@id` string for reproducibility.

In [ ]:
# Replace these IDs with actual ones listed in the previous overview step
mw_field_id = 'cr:MW'       # @id for molecular weight field
desc_field_id = 'cr:Description'  # @id for description/annotation field (example)
if mw_field_id not in main_df.columns:
    print(f"'{mw_field_id}' not found. Available columns: {main_df.columns.tolist()}")
else:
    # Ensure MW is numeric
    main_df[mw_field_id] = pd.to_numeric(main_df[mw_field_id], errors='coerce')

    # Filter proteins for MW > 10000
    threshold = 10000
    filtered_df = main_df[main_df[mw_field_id] > threshold].copy()
    print(f"Filtered {filtered_df.shape[0]} records with {mw_field_id} > {threshold}")
    print(filtered_df[[mw_field_id]].head())

    # Normalize MW
    filtered_df[f"{mw_field_id}_normalized"] = (filtered_df[mw_field_id] - filtered_df[mw_field_id].mean()) / filtered_df[mw_field_id].std()
    print(filtered_df[[mw_field_id, f"{mw_field_id}_normalized"]].head())

    # Group by description field if available
    if desc_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(desc_field_id)[mw_field_id].mean().reset_index().sort_values(mw_field_id, ascending=False)
        print("Average MW by description (first 5):")
        print(grouped.head())

## 5. Visualization
Let's plot the distribution of molecular weights and (if available) the top protein descriptions for the highest MW proteins.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of MW for all proteins
if mw_field_id in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[mw_field_id].dropna(), bins=30, kde=True)
    plt.title('Protein Molecular Weight (MW) Distribution')
    plt.xlabel('MW (Daltons)')
    plt.ylabel('Count')
    plt.show()

    # Plot top 10 proteins by MW
    top10 = main_df.sort_values(mw_field_id, ascending=False).head(10)
    plt.figure(figsize=(10, 4))
    sns.barplot(data=top10, x=mw_field_id, y=desc_field_id if desc_field_id in top10.columns else mw_field_id)
    plt.title('Top 10 proteins by MW')
    plt.xlabel('MW (Daltons)')
    if desc_field_id in top10.columns:
        plt.ylabel('Description')
    plt.show()
else:
    print(f"Field '{mw_field_id}' not found for plotting.")

## 6. Conclusion
- This notebook demonstrates the FAIR² dataset loading, schema exploration, and analysis using the `mlcroissant` library in Python.
- We extracted records using Croissant `@id` references, investigated protein molecular weights, normalized and grouped fields, and visualized data distributions.
- For deeper analysis (e.g., by protein class, modification, or sample origin), refer to additional fields in the dataset schema and their `@id`s listed in Section 2.